*0.1 Python for GenAI · async/await, asyncio.gather, concurrency for API calls*

# async/await

**The situation.** You run a support chatbot. It is a web service: a customer's question comes in, the service asks the model, the answer goes back. One model call takes about 1 second. On Monday morning three customers ask at the same second. This item is about what happens in that second.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

import asyncio
from concurrent.futures import ThreadPoolExecutor


# A notebook already has an event loop running, so asyncio.run() is not allowed here.
# This helper runs the async code on a separate thread instead. In a normal script you
# would simply write asyncio.run(main()).
def run_async(coroutine):
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coroutine).result()

**First, a service written the wrong way.** It is a FastAPI web service with one endpoint. It looks fine: the function is marked `async def`. But it calls the model with a *normal* client, and a normal client makes the whole program stand still while it waits.

In [2]:
from fastapi import FastAPI
from openai import OpenAI

app = FastAPI()
sync_ai = OpenAI(timeout=30)


@app.post("/answer-blocking")
async def answer_blocking(question: str) -> dict:
    # The server stops here until the model answers. Nobody else is served meanwhile.
    reply = sync_ai.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": question}]
    )
    return {"answer": reply.choices[0].message.content}


print("endpoint registered: /answer-blocking")

endpoint registered: /answer-blocking


**Now the same service written the right way.** One word changes: `await`. `await` means "I am waiting for something slow; serve the next customer while I wait." For that to work the client must be the async one, `AsyncOpenAI`, which knows how to hand the wait back.

In [3]:
from openai import AsyncOpenAI

async_ai = AsyncOpenAI(timeout=30)


@app.post("/answer")
async def answer(question: str) -> dict:
    # "await": while the model works, the server is free to take other customers.
    reply = await async_ai.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": question}]
    )
    return {"answer": reply.choices[0].message.content}


print("endpoint registered: /answer")

endpoint registered: /answer


**Start the service.** In production this is `uvicorn app:app`. Here it starts in a background thread so the notebook can call it.

In [4]:
import threading
import time

import httpx
import uvicorn

server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="error"))
threading.Thread(target=server.run, daemon=True).start()
for _ in range(50):  # wait until the port answers
    try:
        httpx.get("http://127.0.0.1:8000/docs", timeout=1)
        break
    except httpx.HTTPError:
        time.sleep(0.2)
print("service is up on http://127.0.0.1:8000")

service is up on http://127.0.0.1:8000


**Now the test: three customers at once.** This code plays the three customers. It sends three questions at the same moment and measures how long until all three answers are back — once against each endpoint.

In [5]:
async def three_customers(path: str) -> float:
    questions = ["How do I reset my password?", "Where is my order?", "Can I get a refund?"]
    async with httpx.AsyncClient(base_url="http://127.0.0.1:8000", timeout=60) as web:
        started = time.perf_counter()
        tasks = []
        for question in questions:
            tasks.append(web.post(path, params={"question": question}))  # all three click "send"
        await asyncio.gather(*tasks)  # wait until all three answers are back
        return time.perf_counter() - started


run_async(three_customers("/answer"))  # one warm-up round so both measurements start warm
blocking_seconds = run_async(three_customers("/answer-blocking"))
async_seconds = run_async(three_customers("/answer"))
print("blocking endpoint:", round(blocking_seconds, 1), "s")
print("async endpoint:   ", round(async_seconds, 1), "s")
server.should_exit = True
assert async_seconds < blocking_seconds

blocking endpoint: 7.6 s
async endpoint:    4.5 s


**Reading the numbers.** The blocking service took about three times longer: customer 1 waited, then customer 2, then customer 3. The async service answered all three in roughly the time of one call: the three waits happened at the same time. Same model, same questions, same code except `await` and the async client.

```
blocking   c1 ■■■■■■■■■■ c2 ■■■■■■■■■■ c3 ■■■■■■■■■■     ≈ 3 calls long
async      c1 ■■■■■■■■■■
           c2 ■■■■■■■■■■                                 ≈ 1 call long
           c3 ■■■■■■■■■■
```

**The rule to remember.** Inside `async def`, every slow thing — the model, the database, another web service — must be called with `await` and an async client. One blocking call anywhere freezes every customer. This bug passes every test (tests send one request at a time) and collapses on the first busy morning.

| Use it when | Don't when | Instead use |
|---|---|---|
| writing web services and background workers that call models, databases or APIs | the work is heavy calculation — it never waits, so nothing overlaps | a process pool for calculation; the plain sync client in scripts where nothing runs in parallel |

**Watch out**
- Forgetting `await` returns a "coroutine object" instead of the answer. Nothing runs, and there is no error.
- Create the `AsyncOpenAI` client once at startup and reuse it; do not build one per request.
- `asyncio.run(...)` starts the whole thing once; every function that waits inside is `async def`.